# **Noam US Stocks Screener Methods**

## **Le Screener**

Actions répondant aux critères suivant :

*   'Country' : 'USA'
*   'Market Cap.' : "+Micro (over $50mln)"
*   'Performance' : 'Month +30%'
*   'Volatility' : 'Week - Over 10%'
*   'Performance 2' : 'Quarter Up'
*   RS : >95



In [103]:
# @title
pip install finvizfinance

In [104]:
# @title
pip show finvizfinance

Name: finvizfinance
Version: 1.2.0
Summary: Finviz Finance. Information downloader.
Home-page: https://github.com/lit26/finvizfinance
Author: Tianning Li
Author-email: ltianningli@gmail.com
License: The MIT License (MIT)
Location: /usr/local/lib/python3.12/dist-packages
Requires: beautifulsoup4, lxml, pandas, requests
Required-by: 


In [105]:
# @title
from finvizfinance.screener.overview import Overview
foverview = Overview()
filters_dict = {'Country' : 'USA', 'Market Cap.' : "+Micro (over $50mln)", 'Performance' : 'Month +10%', 'Volatility' : 'Month - Over 5%', 'Performance 2' : 'Quarter Up'}
foverview.set_filter(filters_dict=filters_dict)
df = foverview.screener_view()

,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume
0,ACRV,Acrivon Therapeutics Inc,Healthcare,Biotechnology,USA,7.447000e+07,NaN,2.36,-0.0288,291688.0
1,AENT,Alliance Entertainment Holding Corporation,Communication Services,Entertainment,USA,3.424300e+08,17.51,6.72,-0.0204,38818.0
2,AGX,"Argan, Inc",Industrials,Engineering & Construction,USA,4.950000e+09,43.17,358.72,0.0417,332635.0
3,ALB,Albemarle Corp,Basic Materials,Specialty Chemicals,USA,1.479000e+10,NaN,125.68,0.0353,3631732.0
4,ALMS,Alumis Inc,Healthcare,Biotechnology,USA,6.858600e+08,NaN,6.57,0.0250,1622566.0


In [106]:
df

,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume
0,ACRV,Acrivon Therapeutics Inc,Healthcare,Biotechnology,USA,7.447000e+07,NaN,2.36,-0.0288,291688.0
1,AENT,Alliance Entertainment Holding Corporation,Communication Services,Entertainment,USA,3.424300e+08,17.51,6.72,-0.0204,38818.0
2,AGX,"Argan, Inc",Industrials,Engineering & Construction,USA,4.950000e+09,43.17,358.72,0.0417,332635.0
3,ALB,Albemarle Corp,Basic Materials,Specialty Chemicals,USA,1.479000e+10,NaN,125.68,0.0353,3631732.0
4,ALMS,Alumis Inc,Healthcare,Biotechnology,USA,6.858600e+08,NaN,6.57,0.0250,1622566.0
...,...,...,...,...,...,...,...,...,...,...
195,XFOR,X4 Pharmaceuticals Inc,Healthcare,Biotechnology,USA,3.191400e+08,NaN,3.65,0.0139,575439.0
196,XMTR,Xometry Inc,Industrials,Industrial Distribution,USA,3.090000e+09,NaN,60.36,0.0344,617740.0
197,ZBIO,Zenas Biopharma Inc,Healthcare,Biotechnology,USA,1.990000e+09,NaN,37.15,0.0260,255945.0
198,ZURA,Zura Bio Ltd,Healthcare,Biotechnology,USA,2.516400e+08,NaN,3.87,0.0104,440813.0


Calculer et ajouter le RS pour ces filtres

In [107]:
# @title
import pandas as pd
import yfinance as yf
import numpy as np
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONSTANTES DES SEUILS (Replay Mode du script)
# ============================================
SEUILS = {
    'first': 195.93,  # RS Score pour 99 (98ème percentile)
    'scnd':  117.11,  # RS Score pour 90 (89ème percentile)
    'thrd':  99.04,   # RS Score pour 70 (69ème percentile)
    'frth':  91.66,   # RS Score pour 50 (49ème percentile)
    'ffth':  80.96,   # RS Score pour 30 (29ème percentile)
    'sxth':  53.64,   # RS Score pour 10 (9ème percentile)
    'svth':  24.86    # RS Score pour 1  (1er percentile)
}

# ============================================
# FONCTION DE CALCUL DU RS SCORE
# ============================================
def calculate_rs_score(ticker_data, spx_data):
    """
    Calcule le RS Score (score de performance relative)
    ticker_data et spx_data doivent être des Series de 252 jours minimum
    """
    try:
        # Vérifier qu'on a assez de données
        if len(ticker_data) < 252 or len(spx_data) < 252:
            return np.nan

        # Récupérer les indices des 4 périodes
        n63  = min(62, len(ticker_data) - 1)
        n126 = min(125, len(ticker_data) - 1)
        n189 = min(188, len(ticker_data) - 1)
        n252 = min(251, len(ticker_data) - 1)

        # Performance du ticker
        perf_ticker_63  = ticker_data.iloc[-1][0] / ticker_data.iloc[-1-n63][0]
        perf_ticker_126 = ticker_data.iloc[-1][0] / ticker_data.iloc[-1-n126][0]
        perf_ticker_189 = ticker_data.iloc[-1][0] / ticker_data.iloc[-1-n189][0]
        perf_ticker_252 = ticker_data.iloc[-1][0] / ticker_data.iloc[-1-n252][0]

        # Performance du SPX
        perf_spx_63  = spx_data.iloc[-1][0] / spx_data.iloc[-1-n63][0]
        perf_spx_126 = spx_data.iloc[-1][0] / spx_data.iloc[-1-n126][0]
        perf_spx_189 = spx_data.iloc[-1][0] / spx_data.iloc[-1-n189][0]
        perf_spx_252 = spx_data.iloc[-1][0] / spx_data.iloc[-1-n252][0]

        # RS pondéré (40% poids sur 63j, 20% sur les autres)
        rs_stock = 0.4 * perf_ticker_63 + 0.2 * perf_ticker_126 + 0.2 * perf_ticker_189 + 0.2 * perf_ticker_252
        rs_ref   = 0.4 * perf_spx_63   + 0.2 * perf_spx_126   + 0.2 * perf_spx_189   + 0.2 * perf_spx_252

        # RS Score
        rs_score = (rs_stock / rs_ref) * 100

        return float(rs_score)
    except:
        return np.nan

# ============================================
# FONCTION D'ATTRIBUTION DU PERCENTILE
# ============================================
def f_attribute_percentile(rs_score, taller_perf, smaller_perf, range_up, range_dn, weight):
    """
    Interpole linéairement pour attribuer un percentile entre deux seuils
    """
    try:
        sum_val = rs_score + (rs_score - smaller_perf) * weight
        if sum_val > taller_perf - 1:
            sum_val = taller_perf - 1

        k1 = smaller_perf / range_dn
        k2 = (taller_perf - 1) / range_up
        k3 = (k1 - k2) / (taller_perf - 1 - smaller_perf)
        rs_rating = sum_val / (k1 - k3 * (rs_score - smaller_perf))

        if rs_rating > range_up:
            rs_rating = range_up
        if rs_rating < range_dn:
            rs_rating = range_dn

        return rs_rating
    except:
        return np.nan

# ============================================
# FONCTION DE CALCUL DU RS RATING (1-99)
# ============================================
def calculate_rs_rating(rs_score):
    """
    Convertit le RS Score en RS Rating (1-99)
    """
    if pd.isna(rs_score):
        return np.nan

    first, scnd, thrd, frth, ffth, sxth, svth = (
        SEUILS['first'], SEUILS['scnd'], SEUILS['thrd'], SEUILS['frth'],
        SEUILS['ffth'], SEUILS['sxth'], SEUILS['svth']
    )

    if rs_score >= first:
        return 99
    if rs_score <= svth:
        return 1
    if rs_score < first and rs_score >= scnd:
        return f_attribute_percentile(rs_score, first, scnd, 98, 90, 0.33)
    if rs_score < scnd and rs_score >= thrd:
        return f_attribute_percentile(rs_score, scnd, thrd, 89, 70, 2.1)
    if rs_score < thrd and rs_score >= frth:
        return f_attribute_percentile(rs_score, thrd, frth, 69, 50, 0)
    if rs_score < frth and rs_score >= ffth:
        return f_attribute_percentile(rs_score, frth, ffth, 49, 30, 0)
    if rs_score < ffth and rs_score >= sxth:
        return f_attribute_percentile(rs_score, ffth, sxth, 29, 10, 0)
    if rs_score < sxth and rs_score >= svth:
        return f_attribute_percentile(rs_score, sxth, svth, 9, 2, 0)

    return np.nan

# ============================================
# FONCTION PRINCIPALE
# ============================================
def add_rs_rating_to_df(df, ticker_column='Ticker', lookback_days=252):
    """
    Ajoute une colonne 'RS_Rating' au dataframe

    Parameters:
    -----------
    df : pandas DataFrame
        Dataframe contenant les tickers
    ticker_column : str
        Nom de la colonne contenant les tickers
    lookback_days : int
        Nombre de jours historiques à récupérer (default: 252 pour 1 an)

    Returns:
    --------
    df : pandas DataFrame
        Dataframe avec la nouvelle colonne 'RS_Rating'
    """

    print("📥 Téléchargement des données SPX...")
    end_date = datetime.now()
    start_date = end_date - timedelta(days=368)

    try:
        spx_data = yf.download('^GSPC', start=start_date, end=end_date, progress=False)['Close']
    except:
        print("❌ Erreur téléchargement SPX")
        return df

    rs_ratings = []
    rs_scores = []

    print(f"📊 Calcul RS Rating pour {len(df)} actions...\n")

    for idx, row in df.iterrows():
        ticker = row[ticker_column].strip()

        try:
            # Télécharger les données du ticker
            ticker_data = yf.download(ticker, start=start_date, end=end_date, progress=False)['Close']

            # Calculer le RS Score
            rs_score = calculate_rs_score(ticker_data, spx_data)


            # Convertir en RS Rating
            rs_rating = calculate_rs_rating(rs_score)

            rs_ratings.append(rs_rating)
            rs_scores.append(rs_score)

            status = f"✓ {ticker}: Score={rs_score:.2f} | Rating={rs_rating:.1f}" if not pd.isna(rs_rating) else f"⚠ {ticker}: Données insuffisantes"
            print(f"{idx+1:3d}. {status}")

        except Exception as e:
            print(f"{idx+1:3d}. ✗ {ticker}: Erreur ({str(e)[:30]})")
            rs_scores.append(np.nan)
            rs_ratings.append(np.nan)

    df['RS_Rating'] = rs_ratings
    df['RS_Score'] = rs_scores

    print(f"\n✅ Calcul terminé!\n")

    return df

In [108]:
# @title
# ============================================
# EXEMPLE D'UTILISATION
# ============================================
df = add_rs_rating_to_df(df, ticker_column='Ticker')


📥 Téléchargement des données SPX...
📊 Calcul RS Rating pour 200 actions...

  1. ✓ ACRV: Score=116.64 | Rating=89.0
  2. ✓ AENT: Score=150.41 | Rating=98.0
  3. ✓ AGX: Score=190.98 | Rating=98.0
  4. ✓ ALB: Score=149.58 | Rating=98.0
  5. ✓ ALMS: Score=112.55 | Rating=87.5
  6. ✓ ALT: Score=87.49 | Rating=41.1
  7. ✓ ALTO: Score=168.96 | Rating=98.0
  8. ✓ AMPY: Score=117.96 | Rating=90.3
  9. ✓ ANNX: Score=141.52 | Rating=98.0
 10. ✓ ANRO: Score=396.89 | Rating=99.0
 11. ✓ ANVS: Score=120.97 | Rating=91.5
 12. ✓ APGE: Score=150.71 | Rating=98.0
 13. ✓ APPN: Score=118.68 | Rating=90.6
 14. ✓ APYX: Score=221.04 | Rating=99.0
 15. ✓ ARMP: Score=296.35 | Rating=99.0
 16. ✓ ARQT: Score=196.04 | Rating=99.0
 17. ✓ ARVN: Score=112.09 | Rating=87.3
 18. ✓ ASMB: Score=204.70 | Rating=99.0
 19. ✓ ATNI: Score=111.30 | Rating=86.9
 20. ✓ AXGN: Score=162.14 | Rating=98.0
 21. ✓ AXTI: Score=476.74 | Rating=99.0
 22. ⚠ BALY: Données insuffisantes
 23. ⚠ BBNX: Données insuffisantes
 24. ✓ BBOT: Score

In [111]:
#classer l'action par RS_Score croissant
df_filtered = df[df['RS_Rating'] > 95]

In [112]:
df_filtered

,Ticker,Company,Sector,Industry,Country,Market Cap,P/E,Price,Change,Volume,RS_Rating,RS_Score
1,AENT,Alliance Entertainment Holding Corporation,Communication Services,Entertainment,USA,3.424300e+08,17.51,6.72,-0.0204,38818.0,98.0,150.413037
2,AGX,"Argan, Inc",Industrials,Engineering & Construction,USA,4.950000e+09,43.17,358.72,0.0417,332635.0,98.0,190.984505
3,ALB,Albemarle Corp,Basic Materials,Specialty Chemicals,USA,1.479000e+10,NaN,125.68,0.0353,3631732.0,98.0,149.583725
6,ALTO,Alto Ingredients Inc,Basic Materials,Specialty Chemicals,USA,1.655200e+08,NaN,2.14,0.0490,1107215.0,98.0,168.957842
8,ANNX,Annexon Inc,Healthcare,Biotechnology,USA,5.876000e+08,NaN,4.06,0.2687,10650536.0,98.0,141.522471
...,...,...,...,...,...,...,...,...,...,...,...,...
193,WDC,Western Digital Corp,Technology,Computer Hardware,USA,5.264000e+10,22.45,153.97,0.0073,6499050.0,99.0,244.183711
196,XMTR,Xometry Inc,Industrials,Industrial Distribution,USA,3.090000e+09,NaN,60.36,0.0344,617740.0,98.0,150.480170
197,ZBIO,Zenas Biopharma Inc,Healthcare,Biotechnology,USA,1.990000e+09,NaN,37.15,0.0260,255945.0,99.0,272.849601
198,ZURA,Zura Bio Ltd,Healthcare,Biotechnology,USA,2.516400e+08,NaN,3.87,0.0104,440813.0,99.0,212.838970


## **Le Money Management, Le Risk Management, Les Prises de Positions (Timing)**

### **Market Timing**

Checker si l'action a vecu ses 3 jambes haussières

Checker si l'entreprise vaut le coup sur Zonebourse (au moins rapidement)

Timing :
* Attendre pendant la consolidation - l'identifier - attendre la cassure
* Etre attentif à la diminution des volumes (moins de vendeurs)
* Attendre le breakout avec volume x1.5 (attention aux fausses cassures)


Gestion de la position
* Securisation
* Prise de profit partiel

Trade Management Systématique
SL sous les consolidations

Sortir de la position

*Il faut qu'on soit capable d'afficher dans une page web les differentes actions de la watchlist en déroulant des flèches et obtenir toujours les mêmes informations, par exemple obtenir les bougies japonaises sur 1 an avec l'evolution du volume et la capacité à rajouter quelques indicateurs techniques. On ajoute alors a une nouvelle liste : Pret à acheter*
